# Actor GraphMAE encoder depth check
Train a two-layer GraphMAE encoder on Actor with seed 0. Run both
column-standardized and binary input features. Keep the one-layer decoder
and the other Actor hyperparameters fixed. Evaluate cosine on all unique
cross-label edges used in the Figure 8 reproduction.


In [ ]:
MODE = "configure"  # "configure", "smoke" or "full"
SEED = 0


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

assert MODE in {"configure", "smoke", "full"}
work = Path("/tmp/dgmae-figure8-actor-depth")
checkout = work / "source"
graphmae = work / "graphmae"
experiment_commit = "736ff4e4bc5794323f1d565f8c3a578d40b88134"
graphmae_commit = "b14f080c919257b495e3cb6474286d5252d6a635"

def clone(url, commit, destination):
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", url, commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)

if MODE != "configure":
    subprocess.run(["nvidia-smi", "-L"], check=True)
    clone("https://github.com/phud774/CS313.R11.git", experiment_commit, checkout)
    clone("https://github.com/THUDM/GraphMAE.git", graphmae_commit, graphmae)
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
    setup_dir = checkout / "experiments" / "ablation"
    subprocess.run(["bash", str(setup_dir / "setup_kaggle.sh"), str(work)], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(work / ".dgmae-venv/bin/python"),
                    "matplotlib==3.9.4", "tensorboardX==2.6.2.2"], check=True)


In [ ]:
if MODE != "configure":
    from IPython.display import Image, display

    script = checkout / "DGMAE-REIMPL/experiments/figure8/run_figure8.py"
    output = Path("/kaggle/working/actor-graphmae-depth")
    env = dict(os.environ, PYTHONUNBUFFERED="1", DGLBACKEND="pytorch", OMP_NUM_THREADS="4")
    stages = ("smoke", "full") if MODE == "full" else ("smoke",)
    for stage in stages:
        for features in ("standard", "raw"):
            destination = output / stage / features
            cmd = [
                "bash", str(setup_dir / "run_kaggle_python.sh"), str(work), "-B", str(script),
                "--graphmae", str(graphmae), "--data", str(work / "datasets"),
                "--output", str(destination), "--datasets", "actor", "--seed", str(SEED),
                "--models", "graphmae", "--graphmae-layers", "2", "--graphmae-features", features,
            ]
            if stage == "smoke":
                cmd.append("--smoke")
            print(f"Actor GraphMAE / two layers / {features} / {stage}", flush=True)
            subprocess.run(cmd, env=env, check=True)
            display(Image(filename=str(destination / "figure8.png")))
